In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType, StringType
import os

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("2_Ingest_MovieLens_To_HDFS")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.driver.host", "notebook")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "1")
    .config("spark.executor.instances", "1")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

LOCAL_RAW_DIR = "/workspace/data/raw/ml-25m"
LOCAL_RAW = f"file://{LOCAL_RAW_DIR}"
HDFS_BASE = "hdfs://namenode:8020/netflix-recsys"
HDFS_RAW = f"{HDFS_BASE}/raw/ml-25m"

print("Spark version:", spark.version)
print("Local raw dir:", LOCAL_RAW_DIR)
print("Local files:", sorted(os.listdir(LOCAL_RAW_DIR)))

In [ ]:
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", LongType(), True),
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True),
])

links_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("imdbId", StringType(), True),
    StructField("tmdbId", StringType(), True),
])

genome_scores_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("tagId", IntegerType(), True),
    StructField("relevance", FloatType(), True),
])

genome_tags_schema = StructType([
    StructField("tagId", IntegerType(), True),
    StructField("tag", StringType(), True),
])

In [ ]:
ratings = spark.read.csv(f"{LOCAL_RAW}/ratings.csv", header=True, schema=ratings_schema)
movies = spark.read.csv(f"{LOCAL_RAW}/movies.csv", header=True, schema=movies_schema)
tags = spark.read.csv(f"{LOCAL_RAW}/tags.csv", header=True, schema=tags_schema)
links = spark.read.csv(f"{LOCAL_RAW}/links.csv", header=True, schema=links_schema)
genome_scores = spark.read.csv(f"{LOCAL_RAW}/genome-scores.csv", header=True, schema=genome_scores_schema)
genome_tags = spark.read.csv(f"{LOCAL_RAW}/genome-tags.csv", header=True, schema=genome_tags_schema)

counts = {
    "ratings": ratings.count(),
    "movies": movies.count(),
    "tags": tags.count(),
    "links": links.count(),
    "genome_scores": genome_scores.count(),
    "genome_tags": genome_tags.count(),
}
counts

In [ ]:
ratings.printSchema()
movies.printSchema()
genome_scores.printSchema()
genome_tags.printSchema()

In [ ]:
ratings.write.mode("overwrite").parquet(f"{HDFS_RAW}/ratings")
movies.write.mode("overwrite").parquet(f"{HDFS_RAW}/movies")
tags.write.mode("overwrite").parquet(f"{HDFS_RAW}/tags")
links.write.mode("overwrite").parquet(f"{HDFS_RAW}/links")
genome_scores.write.mode("overwrite").parquet(f"{HDFS_RAW}/genome_scores")
genome_tags.write.mode("overwrite").parquet(f"{HDFS_RAW}/genome_tags")

In [ ]:
from pyspark.sql import Row

fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
paths = [
    f"{HDFS_BASE}/bronze",
    f"{HDFS_BASE}/silver",
    f"{HDFS_BASE}/gold",
    f"{HDFS_BASE}/models",
    f"{HDFS_BASE}/outputs",
]

results = []
for path in paths:
    hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(path)
    created_or_exists = fs.mkdirs(hdfs_path)
    exists = fs.exists(hdfs_path)
    results.append(Row(path=path, created_or_exists=bool(created_or_exists), exists=bool(exists)))

spark.createDataFrame(results).show(truncate=False)

In [ ]:
spark.stop()